# Cross-dataset check

One Cambridge epoch file and one NEMAR EDF go through the same loader and the same feature call. They are not pooled into one inferential test. Both files have to be downloaded first. See `docs/validation/DATASETS.md`. Set `EEGMATH_DATA` if they are not under `external_data/`.

In [1]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display
from eegmath import extract_features, load
from eegmath.errors import DataError
from eegmath.recording import EEGCollection, align_channels

def _validation_data() -> Path:
    override = os.environ.get('EEGMATH_DATA')
    if override:
        return Path(override).expanduser()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'external_data').is_dir():
            return candidate / 'external_data'
    return here / 'external_data'

DATA = _validation_data()
cambridge_dir = DATA / 'cambridge_propofol' / 'extracted' / 'Sedation-RestingState'
nemar_path = DATA / 'nemar_nm000109' / 'sub-00' / 'eeg' / 'sub-00_task-rest_eeg.edf'
assert cambridge_dir.is_dir() and nemar_path.is_file(), (
    f'Recordings were not found under {DATA}. See docs/validation/DATASETS.md.'
)
cambridge_path = sorted(cambridge_dir.glob('*.set'))[0]
cambridge = load(cambridge_path).spawn('label', subject_id='cambridge-example', condition='file')
nemar = load(nemar_path)
collection = EEGCollection.from_sequence([cambridge, nemar])
report = collection.compatibility()
display(pd.Series({
    'sfrequencies_hz': report['sfrequencies_hz'],
    'same_sfreq': report['same_sfreq'],
    'units': report['units'],
    'n_shared_channels': report['n_shared_channels'],
    'shared_preview': report['shared_channels'][:12],
}))
assert report['same_sfreq'] is False
try:
    collection.require(same_sfreq=True, same_channels=True)
    raise AssertionError('Mismatched recordings were accepted as one homogeneous collection.')
except DataError as exc:
    print(exc)
shared = align_channels(collection)
assert shared[0].sfreq != shared[1].sfreq
assert shared[0].ch_names == shared[1].ch_names
left = extract_features(shared[0].pick_channels(list(shared[0].ch_names[:4])), ['rms', 'alpha_relative'], reduce_epochs=True)
right = extract_features(shared[1].pick_channels(list(shared[1].ch_names[:4])), ['rms', 'alpha_relative'])
assert {'rms', 'alpha_relative'} <= set(left.columns) and {'rms', 'alpha_relative'} <= set(right.columns)
print('same feature columns, separate recordings, no pooled test')

sfrequencies_hz                                         [250.0, 500.0]
same_sfreq                                                       False
units                                                              [V]
n_shared_channels                                                   19
shared_preview       [C3, C4, Cz, F3, F4, F7, F8, Fp1, Fp2, Fz, O1,...
dtype: object

This collection cannot be treated as homogeneous. sampling frequencies differ: [250.0, 500.0]; channel names or channel order are not identical across recordings. Resample, convert units, or pick a shared channel set explicitly.
same feature columns, separate recordings, no pooled test
